In [1]:
import os
import time
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, DataCollatorForTokenClassification,
    DataCollatorWithFlattening
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import (accuracy_score,classification_report, 
confusion_matrix, balanced_accuracy_score, f1_score)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modelo de GPU: {torch.cuda.get_device_name(0)}")


# Modo offline completo para solucionar problemas al tratar de cargar la configuracion LORA
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"  

GPU disponible: True
Modelo de GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
#Extracción de rutas para la lectura y derivación de archivos
PROJECT_ROOT = Path(os.getcwd()).parent

# Ruta completa a carpetas
MODELOS_PATH = PROJECT_ROOT / 'modelos'
DATOS_PATH = PROJECT_ROOT / 'data' / 'limpieza_final' / 'etiquetado_humano_unificado.csv'
MODELO_TUNEADO_PATH = PROJECT_ROOT / 'modelos'/ 'Modelo_Fold_3'
print(str(MODELOS_PATH))
print(str(MODELO_TUNEADO_PATH))
print(str(DATOS_PATH))
corpus = pd.read_csv(DATOS_PATH, usecols = ['comentario', 'etiquetado_humano','categoria'],encoding = 'utf-8-sig')
print(corpus.head())

C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos
C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos\Modelo_Fold_3
C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\data\limpieza_final\etiquetado_humano_unificado.csv
                                          comentario  etiquetado_humano  \
0  cuál es el más cercado para rayar el nombre de...                3.0   
1  esos baños deberian estar en el metro no saben...                2.0   
2                           no pues bueno me da risa                3.0   
3                                los van a abandonar                2.0   
4                           nada los tiene contentos                4.0   

         categoria  
0  infraestructura  
1  infraestructura  
2  infraestructura  
3  infraestructura  
4  infraestructura  


In [3]:
#Nos quedamos unicamente con los datos que tienen etiqueta, pues estamos
#realizando un aprendizaje supervisado
muestra = corpus[np.isnan(corpus['etiquetado_humano']) == False]

#Se renombran la columnas columnas 'etiquetado_humano' y 'comentario' 
#como 'labels' y 'text' como requerimiento para hacer el fine tuning
muestra.rename(columns = {'etiquetado_humano':'labels','comentario':'text'}, inplace = True)
muestra['labels'] = [int(x-1) for x in muestra['labels']]

#Para evitar problemas de correlacion (inducidos por las categorias),
#hacemos un reordenamiento aleatorio de nuestros datos
muestra = muestra.sample(n = len(muestra['text']))
muestra_ds = Dataset.from_pandas(muestra)


print(muestra.head())

semilla = 61298

                                                  text  labels  \
108  como los baños de la línea 12 que están clausu...       2   
11   les digo que hay goteras en pino suárez llueve...       1   
968  si tan solo usaran toda esa organización y pod...       0   
56   si el metro no sirve imagínate los baños hay n...       1   
129  apoco si creían que morena iba a hacer obras d...       0   

           categoria  
108  infraestructura  
11   infraestructura  
968        seguridad  
56   infraestructura  
129  infraestructura  


In [6]:
modelo = 'nlptown/bert-base-multilingual-uncased-sentiment'
# Primero cargamos el modelo base (sin fine-tuning)
clasificador = AutoModelForSequenceClassification.from_pretrained(
    modelo,
    num_labels = 5,
    id2label= {0:'Negativo',1:'Parcialmente Negativo', 2:'Neutral',3:'Parcialmente Positivo', 4:'Postivo'},
    label2id={'Negativo':0,'Parcialmente Negativo':1, 'Neutral': 2, 'Parcialmente Positivo': 3, 'Positivo':4}
)

# Luego cargamos los adaptadores LoRA (el resultado del tuneo)


clasificador_tuneado = PeftModel.from_pretrained(clasificador,
                                                 str(MODELO_TUNEADO_PATH),
                                                 local_files_only = True,
                                                 config=None,
                                                 cache_dir=None,
                                                 ignore_mismatched_sizes=True,
                                                 force_download=False,
                                                 use_auth_token=None)

# Cargamos el tokenizador de nuestro modelo tuneado
#tokenizador = AutoTokenizer.from_pretrained(str(MODELO_TUNEADO_PATH))

# Movemos nuestro modelo a GPU
#clasificador_tuneado = clasificador_tuneado.to(device)

print("Modelo tuneado listo")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos\Modelo_Fold_3'.

In [5]:
# 1. Cargar modelo base
modelo_base = AutoModelForSequenceClassification.from_pretrained(
    "nlptown/bert-base-multilingual-uncased-sentiment",
    num_labels=5
)

# 2. Leer configuración de LoRA manualmente
config_path = MODELO_TUNEADO_PATH / "adapter_config.json"
with open(config_path, 'r') as f:
    config_dict = json.load(f)

# 3. Crear LoraConfig desde el diccionario
lora_config = LoraConfig(
    task_type=config_dict.get('task_type', 'SEQ_CLS'),
    r=config_dict.get('r', 8),
    lora_alpha=config_dict.get('lora_alpha', 32),
    lora_dropout=config_dict.get('lora_dropout', 0.1),
    target_modules=config_dict.get('target_modules', ['query', 'value']),
    bias=config_dict.get('bias', 'none'),
)

# 4. Cargar pesos del adaptador
adapter_weights_path = MODELO_TUNEADO_PATH / "adapter_model.bin"
if not adapter_weights_path.exists():
    adapter_weights_path = MODELO_TUNEADO_PATH / "adapter_model.safetensors"

# 5. Crear modelo Peft y cargar pesos manualmente
modelo_lora = get_peft_model(clasificador, lora_config)

# Cargar los pesos guardados
state_dict = torch.load(adapter_weights_path, map_location='cpu')
modelo_lora.load_state_dict(state_dict, strict=False)

# Cargar tokenizador
tokenizer = AutoTokenizer.from_pretrained(str(ruta_modelo))

print("✅ Modelo cargado manualmente")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\inqui\\OneDrive\\Desktop\\Clases\\26-2\\LLM_PROJECT_1\\modelos\\Modelo_Fold_3\\adapter_model.safetensors'

In [7]:
class_weights = calcular_class_weights(fold_bueno['train'], num_labels=5)

clasificador = AutoModelForSequenceClassification.from_pretrained(
            modelo,
            num_labels=5,
            id2label={0:'Negativo',1:'Parcialmente Negativo', 2:'Neutral',3:'Parcialmente Positivo', 4:'Positivo'},
            label2id={'Negativo':0,'Parcialmente Negativo':1, 'Neutral': 2, 'Parcialmente Positivo': 3, 'Positivo':4}
        )

        # Configuramos LORA
lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=32,
            lora_alpha=64,
            lora_dropout=0.2,
            bias='lora_only',
            target_modules=['query', 'value', 'key', 'dense'],
        )
        
modelo_lora = get_peft_model(clasificador, lora_config)
if device == 'cuda':
    modelo_lora = modelo_lora.to('cuda')
        
        #Usar WeightedLossTrainer en lugar de Trainer estándar
trainer = WeightedLossTrainer(
            model=modelo_lora,
            args=training_args,
            train_dataset=fold_bueno['train'],
            eval_dataset=fold_bueno['validation'],
            data_collator=colador,
            compute_metrics=metricas,
            class_weights=class_weights,  # ← ¡Aquí pasamos los pesos!
        )
        
# Entrenar
trainer.train()
        
# Evaluar
eval_results = trainer.evaluate()

'cuda'

In [8]:
ruta_modelo = Path("C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/modelos/Modelo_Fold_3")

print("=== DIAGNÓSTICO ===")
print(f"Ruta existe: {ruta_modelo.exists()}")
print(f"Ruta absoluta: {ruta_modelo.absolute()}")
print(f"Como string: {str(ruta_modelo)}")
print(f"Como repr: {repr(str(ruta_modelo))}")

print("\n=== ARCHIVOS EN CARPETA ===")
if ruta_modelo.exists():
    for archivo in ruta_modelo.iterdir():
        print(f"  - {archivo.name}")
    
    # Verificar archivos específicos
    print(f"\nadapter_config.json: {(ruta_modelo / 'adapter_config.json').exists()}")
    print(f"adapter_model.bin: {(ruta_modelo / 'adapter_model.bin').exists()}")
    print(f"adapter_model.safetensors: {(ruta_modelo / 'adapter_model.safetensors').exists()}")
else:
    print("¡ERROR: La ruta no existe!")

# Probar la solución más simple
try:
    from peft import AutoPeftModelForSequenceClassification
    
    print("\n=== INTENTANDO CON AutoPeftModel ===")
    modelo = AutoPeftModelForSequenceClassification.from_pretrained(
        str(ruta_modelo),
        local_files_only=True
    )
    print("✅ ÉXITO!")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

=== DIAGNÓSTICO ===
Ruta existe: True
Ruta absoluta: C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos\Modelo_Fold_3
Como string: C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos\Modelo_Fold_3
Como repr: 'C:\\Users\\inqui\\OneDrive\\Desktop\\Clases\\26-2\\LLM_PROJECT_1\\modelos\\Modelo_Fold_3'

=== ARCHIVOS EN CARPETA ===
  - adapter_config.json
  - metricas.json
  - README.md

adapter_config.json: True
adapter_model.bin: False
adapter_model.safetensors: False

=== INTENTANDO CON AutoPeftModel ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

❌ Error: HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos\Modelo_Fold_3'.


In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pathlib import Path

ruta = Path("C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/modelos/Modelo_Fold_3")

# Cargar como modelo normal (no PeftModel)
modelo = AutoModelForSequenceClassification.from_pretrained(
    str(ruta),  # Ruta simple, sin URI
    local_files_only=True
)

tokenizer = AutoTokenizer.from_pretrained(str(ruta))

print("✅ Modelo cargado correctamente")
print(f"Tipo de modelo: {type(modelo)}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FileNotFoundError: No such file or directory: C:\Users\inqui\OneDrive\Desktop\Clases\26-2\LLM_PROJECT_1\modelos\Modelo_Fold_3\adapter_model.safetensors

ImportError: cannot import name '_maybe_shard_sate_dict_fot_tp' from 'peft.utils.save_and_load' (C:\Users\inqui\AppData\Local\Programs\Python\Python313\Lib\site-packages\peft\utils\save_and_load.py)